# Review and label SpecMine repositories

Load the full SpecMine metadata parquet, choose one repository that is absent from both label files, then select and download exactly one random spec from that repository. The notebook shows that spec's precomputed content features and raw Markdown; files are cached in `notebooks/data` and Markdown is escaped so remote images and embedded HTML are not rendered. Record one row in `examples/real_labels.csv` as `repo_id,spec_id,label,confidence`. Use `1` if AI helped at any point, `0` if it did not, and leave label and confidence blank if evidence is ambiguous. Confidence is a manual score from 0 to 100.

The notebook fails if repositories in `real_labels.csv` and `fake_labels.csv` overlap, and excludes repositories already in either file from sampling. Repository identity is SpecMine's numeric `repo_id`; `spec_id` is retained to join the exact selected spec to its content features and to remain compatible with training. The displayed features are content-based only. Commit history is not used; do not infer authorship from commit metadata.

In [ ]:
import html
import sys
from pathlib import Path

from IPython.display import HTML, display

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "specmine").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))
from specmine.data import download_spec_files, load_data, load_feature_data
from specmine.labeling import match_review_features, select_review_spec
from specmine.labels import read_labels_csv

print("Project source loaded.")

In [ ]:
SEED = 45
DATA_DIR = project_root / "notebooks" / "data"
REAL_LABELS_CSV = project_root / "examples" / "real_labels.csv"
FAKE_LABELS_CSV = project_root / "examples" / "fake_labels.csv"

specs = load_data(curated_only=False)
if specs is None:
    raise RuntimeError("SpecMine data loading failed; no specs are available to review.")
if specs.empty:
    raise ValueError("The SpecMine dataset contains no specs to review.")
required_columns = {"repo_id", "repo_name", "file_url_sha16", "file_path", "file_url"}
missing_columns = required_columns - set(specs.columns)
if missing_columns:
    raise KeyError(f"SpecMine data is missing columns: {sorted(missing_columns)}")

real_labels = read_labels_csv(REAL_LABELS_CSV)
fake_labels = read_labels_csv(FAKE_LABELS_CSV)
review_spec = select_review_spec(specs, real_labels, fake_labels, seed=SEED)
downloaded_spec = download_spec_files(review_spec, data_dir=DATA_DIR)
if len(downloaded_spec) != 1:
    raise RuntimeError("The selected spec could not be downloaded; no review sample is available.")
review_spec = downloaded_spec
feature_rows = load_feature_data(review_spec)
review_features = match_review_features(review_spec, feature_rows)


In [ ]:
row = review_spec.iloc[0]
print(f"Repository: {row['repo_name']}")
print(f"repo_id: {row['repo_id']}")
print(f"Spec file: {row['file_path']}")
print(f"spec_id: {row['file_url_sha16']}")
display(HTML("<h3>Precomputed content features</h3>"))
display(review_features)

In [ ]:
spec_path = Path(row.local_path)
if not spec_path.is_file():
    raise FileNotFoundError("The selected spec file is missing from the data directory.")
spec_markdown = spec_path.read_text(encoding="utf-8")
display(HTML("<h3>Selected spec Markdown</h3>"))
display(HTML("<pre>" + html.escape(spec_markdown) + "</pre>"))
print(
    "After manual review, add exactly one row to real_labels.csv as "
    "repo_id,spec_id,label,confidence. Label must be 0, 1, or blank; "
    "confidence must be 0-100 for a nonblank label (or blank when ambiguous)."
)